In [1]:
import torch

from torch import Tensor
from typing import *

from torch_scatter import scatter

/home/hwj/.miniconda3/envs/pyg/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def act_blks(act_mask: Tensor, blk_size: int) -> Tensor:
    assert act_mask.dtype == torch.bool
    assert act_mask.dim() == 1
    n = act_mask.size(0)
    m = (n + blk_size - 1) // blk_size * blk_size

    blk_mask = act_mask.detach()
    if n != m:
        blk_mask = torch.zeros(m, dtype=act_mask.dtype, device=act_mask.device)
        blk_mask[:n] = act_mask.detach()
    return blk_mask.view(-1, blk_size).max(dim=-1).values

In [3]:
act_mask = torch.rand(1000) < 0.01
blk_mask = act_blks(act_mask, 64)

In [4]:
cached_blocks = torch.randn(blk_mask.size(0), 64, 8)
blks = cached_blocks[blk_mask]

In [5]:
def gather_block(blks: Tensor, blk_mask: Tensor, index: Tensor) -> Tensor:
    blk_size = blks.size(1)
    blk_mask = blk_mask.type(torch.long).cumsum_(dim=0) * blk_mask - 1
    
    idx0 = index.div(blk_size, rounding_mode="floor")
    ind_mask = blk_mask[idx0] >= 0
    
    idx0 = blk_mask[idx0][ind_mask]
    idx1 = index[ind_mask] % blk_size

    s = index.shape[:1] + blks.shape[2:]
    data = torch.zeros(s, dtype=blks.dtype, device=blks.device)
    data[ind_mask] = blks[idx0, idx1, ...]
    return data

In [6]:
def scatter_block(x: Tensor, index: Tensor, blk_mask: Tensor, blk_size: int, reduce: str = "sum") -> Tensor:
    num_blks = blk_mask.count_nonzero()
    blk_mask = blk_mask.type(torch.long).cumsum_(dim=0) * blk_mask - 1

    idx0 = index.div(blk_size, rounding_mode="floor")
    ind_mask = blk_mask[idx0] >= 0

    idx0 = blk_mask[idx0][ind_mask]
    idx1 = index[ind_mask] % blk_size

    data = scatter(
        src=x[ind_mask],
        index=idx0*blk_size+idx1,
        dim=0,
        dim_size=num_blks*blk_size,
        reduce=reduce,
    )
    return data.view(num_blks, blk_size, *x.shape[1:])

In [11]:
# def index_block(index: Tensor, blk_mask: Tensor, blk_size: int) -> Tensor:
    

In [8]:
index = torch.randint(1000, size=(100,), dtype=torch.long)
(gather_block(blks, blk_mask, index)**2).sum(dim=-1) != 0

tensor([False,  True,  True, False,  True, False, False,  True, False,  True,
        False,  True,  True, False, False, False, False, False, False, False,
        False,  True,  True, False, False, False,  True, False, False,  True,
         True, False, False,  True,  True, False,  True,  True, False, False,
         True, False,  True, False, False, False, False, False, False, False,
         True,  True, False, False, False,  True,  True,  True, False, False,
         True, False,  True, False,  True, False,  True, False, False,  True,
        False,  True,  True,  True, False, False, False, False, False,  True,
        False,  True, False,  True,  True,  True, False,  True, False,  True,
        False,  True, False,  True,  True, False, False,  True, False,  True])

In [9]:
x = torch.randn(100, 8)
scatter_block(x, index, blk_mask, 64).size()

torch.Size([8, 64, 8])

In [12]:
t = scatter_block(x, index, blk_mask, 64)
t = t.view(-1, *t.shape[2:])
act_blks((t**2).sum(dim=-1) != 0, 64)

tensor([True, True, True, True, True, True, True, True])

In [14]:
index.sort()

torch.return_types.sort(
values=tensor([  5,  15,  18,  19,  25,  48,  62,  64,  68,  76,  88,  92,  94, 106,
        114, 115, 120, 122, 126, 131, 135, 165, 169, 169, 177, 192, 214, 215,
        245, 249, 255, 269, 282, 286, 294, 299, 334, 358, 359, 372, 401, 403,
        430, 437, 455, 459, 459, 459, 471, 484, 490, 504, 505, 516, 528, 529,
        559, 584, 613, 614, 614, 624, 629, 637, 638, 674, 679, 701, 706, 735,
        742, 746, 748, 750, 759, 774, 774, 776, 778, 787, 803, 825, 831, 834,
        861, 862, 868, 877, 882, 889, 899, 909, 921, 933, 938, 954, 960, 969,
        981, 986]),
indices=tensor([ 6, 77, 14, 41, 43,  0, 13, 53, 38, 54, 15, 76, 98, 47, 75, 80,  8, 95,
        17, 34, 83, 87, 62, 93, 81, 20, 67, 39, 32,  5, 70,  4, 89, 72, 37, 64,
        94, 42, 57, 29, 56, 21, 60,  7, 45, 59, 86, 96, 49, 92, 28, 88, 24, 68,
         3, 61, 44,  2, 79, 55, 85, 33, 12,  1, 11, 97, 50,  9, 84, 40, 71, 22,
        26, 30, 73, 16, 90, 27, 19, 35, 48, 18, 46, 74, 65, 78, 52, 31, 23